[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py/syn_09_chaos_fractals.ipynb)


# 09. Deterministic chaos, maps, and fractal structure

_Graduate course: Introduction to Synergetics · English draft_

> **Guiding question:** How can deterministic equations generate bounded yet predictively unstable motion?

**Notebook contract.** Run from top to bottom. All parameters are dimensionless unless stated otherwise. Explain every diagnostic you use; a visually attractive trajectory is not by itself evidence of self-organization.


## Learning outcomes

By the end of this module, you should be able to:

- explain the Lorenz variables and parameters as a truncation of thermal convection
- distinguish sensitive dependence from numerical error and stochasticity
- estimate a largest Lyapunov exponent from paired trajectories
- connect return maps, invariant measures, and fractal geometry


## Prerequisite bridge: flows, attractors, and Lyapunov growth

An autonomous ODE generates a continuous-time **flow** through state space; an iterated rule generates a discrete-time **map**. After transients, trajectories may approach an attractor: a fixed point, periodic orbit, quasiperiodic set, or chaotic invariant set. The basin of attraction is the set of initial states that approach it.

Chaos means deterministic aperiodic dynamics with sensitive dependence, not merely complicated appearance. A Lyapunov exponent measures the long-time exponential rate of infinitesimal perturbation growth. Positive growth must be estimated with repeated renormalization: two freely evolving trajectories eventually saturate at the attractor size and no longer reveal the local exponent. Solver tolerances and step sizes must be varied to distinguish dynamics from numerical error.


## Lorenz flow: thermal convection reduced to three modes

Lorenz obtained the system by a severe Galerkin truncation of the Oberbeck–Boussinesq equations for Rayleigh–Bénard convection: a horizontal fluid layer heated from below. In the nondimensional model, $x$ is proportional to the amplitude and orientation of the convective roll velocity, $y$ measures a horizontal temperature contrast coupled to that roll, and $z$ measures distortion of the vertical temperature profile away from conduction. Thus the two lobes of the attractor correspond to episodes dominated by opposite circulation orientations; they are not two geographic weather regimes.

$$\dot x=\sigma(y-x),\quad \dot y=x(\rho-z)-y,\quad \dot z=xy-\beta z.$$

Here $\sigma$ is the Prandtl number, $\rho=Ra/Ra_c$ is a reduced Rayleigh control parameter, and $\beta$ is fixed by the retained spatial wavenumber and cell geometry. For $\rho<1$, the conductive equilibrium is stable. At $\rho=1$ it loses stability and, for $\rho>1$, two symmetry-related convective equilibria appear:

$$(x_*,y_*,z_*)=\left(\pm\sqrt{\beta(\rho-1)},\,\pm\sqrt{\beta(\rho-1)},\,\rho-1\right).$$

At the classical parameters $(\sigma,\rho,\beta)=(10,28,8/3)$ these steady rolls are unstable and the trajectory switches irregularly between their neighborhoods. The divergence is $-(\sigma+1+\beta)<0$, so phase volumes contract. Yet a positive largest Lyapunov exponent $\lambda_1$ produces local perturbation growth $\|\delta x(t)\|\sim e^{\lambda_1t}\|\delta x(0)\|$. Boundedness plus stretching and folding produces a strange attractor rather than unbounded divergence. The model is a conceptual laboratory for deterministic predictability, not a quantitative weather simulator.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

from scipy.integrate import solve_ivp

sigma, rho, beta = 10.0, 28.0, 8/3
def lorenz(t, state):
    x, y, z = state
    return sigma*(y-x), x*(rho-z)-y, x*y-beta*z

t = np.linspace(0, 60, 12000)
y0 = np.array([1.0, 1.0, 1.0])
eps0 = 1e-8
s1 = solve_ivp(lorenz, (0, 60), y0, t_eval=t, rtol=1e-10, atol=1e-12)
s2 = solve_ivp(lorenz, (0, 60), y0 + [eps0, 0, 0], t_eval=t, rtol=1e-10, atol=1e-12)
distance = np.linalg.norm(s2.y-s1.y, axis=0)

convective = np.sqrt(beta*(rho-1))
print("conductive equilibrium:", (0.0, 0.0, 0.0))
print("convective equilibria:", (convective, convective, rho-1),
      (-convective, -convective, rho-1))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
mask = t > 10
ax[0].plot(s1.y[0, mask], s1.y[2, mask], lw=0.35)
ax[0].set(xlabel="x", ylabel="z", title="Lorenz attractor projection")
ax[1].semilogy(t, distance)
ax[1].axhline(1, color="k", ls="--", lw=1)
ax[1].set(xlabel="time", ylabel="trajectory separation", title="Growth and saturation")
plt.tight_layout()


In [ ]:
# Logistic-map bifurcation diagram and Lyapunov exponent.
r_values = np.linspace(2.8, 4.0, 1400)
x = 0.37*np.ones_like(r_values)
for _ in range(1200):
    x = r_values*x*(1-x)

rs, xs, lambdas = [], [], np.zeros_like(r_values)
for _ in range(260):
    x = r_values*x*(1-x)
    rs.append(r_values.copy())
    xs.append(x.copy())
    lambdas += np.log(np.maximum(np.abs(r_values*(1-2*x)), 1e-14))
lambdas /= 260

fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(np.concatenate(rs), np.concatenate(xs), ",k", alpha=0.18)
ax[0].set(ylabel="asymptotic x", title="Logistic-map bifurcation diagram")
ax[1].plot(r_values, lambdas, lw=0.8)
ax[1].axhline(0, color="k", lw=1)
ax[1].set(xlabel="control r", ylabel=r"Lyapunov exponent $\lambda$")
plt.tight_layout()


## Assessed exercise

First verify the conductive and convective equilibria and their stability on both sides of $ho=1$, interpreting the result as onset of convection. Then implement the standard renormalization algorithm for the Lorenz largest Lyapunov exponent at $ho=28$. Demonstrate a fit window with exponential growth and show robustness to renormalization interval, integrator tolerance, and discarded transient.

Your submission must include a claim, the mathematical or physical reason for it, numerical evidence, and one limitation.


## Reading and attribution

**Local course reading:** Mikhailov and Loskutov, *Foundations of Synergetics II: Complex Patterns*; Strogatz, *Nonlinear Dynamics and Chaos*.

**Verified web reading:**
- [Lorenz's original paper](https://doi.org/10.1175/1520-0469(1963)020%3C0130:DNF%3E2.0.CO;2)
- [Scholarpedia: butterfly effect](https://www.scholarpedia.org/article/Butterfly_effect)
- [Scholarpedia: chaos topology](https://www.scholarpedia.org/article/Chaos_topology)

The local books are course-provided sources. Web links are supplementary and were verified on 2026-09-04.
